# sum-and-broadcast-duality composite — cx27: sum_back broadcasts; unbroadcast_back collapses — dual ops

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `sum-and-broadcast-duality`, `unbroadcast-pattern`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "sum-and-broadcast-duality"
DD_ATOM_IDS = ["sum-and-broadcast-duality", "unbroadcast-pattern"]
DD_SUBTOPICS = ["Backprop: sum/broadcast duality", "Backprop: Unbroadcast pattern"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Forward `sum(x, dim)` reduces — backward must broadcast the upstream grad back to `x.shape`. Forward `broadcast_to(x, shape)` expands — backward must SUM the upstream grad back down to `x.shape`. They are exact duals: one inserts a size-1 axis and expands, the other peels leading axes and sums out size-1 axes (the unbroadcast pattern).

Composing them means: take a tensor `x`, run forward `sum(dim, keepdim=False)`, then verify that the gradient round-trip — sum_back to expand the upstream grad back to `x.shape`, AND unbroadcast to collapse a broadcast back to `x.shape` — agree on the same shape and act as exact inverses on size-1 axes.

### Composite Exercise — sum_back broadcasts; unbroadcast_back collapses — dual ops

**Atoms exercised together**: `sum-and-broadcast-duality`, `unbroadcast-pattern`

Implement `cx27_round_trip(x, dim)` that returns a dict with FOUR tensors:

- `'forward_sum'` — `x.sum(dim=dim, keepdim=False)`
- `'sum_back_grad'` — the gradient w.r.t. `x` of `forward_sum.sum()` (i.e. seed `grad_out = ones_like(forward_sum)`, then sum_back to `x.shape`). Use unsqueeze+expand+clone.
- `'unbroadcast_grad'` — call `unbroadcast(t.ones_like(x), x)` (no-op since shapes match). After the round trip, this should match `sum_back_grad` and both should be `ones_like(x)`.
- `'peel_grad'` — the LOAD-BEARING case for `unbroadcast`. Construct a fixed test fixture INSIDE the function: a reference shape `peel_ref = t.zeros(5, 1, 4)` and an upstream gradient `peel_grad_out = t.ones(5, 3, 4)` (i.e. the broadcast-back of a (5,1,4) tensor along axis 1). Call your `unbroadcast(peel_grad_out, peel_ref)`. The result must have shape `(5, 1, 4)` — `unbroadcast` must genuinely SUM along axis 1 (peeling 3 → 1), not pass the input through. Each entry of the result should be 3.0 (sum of three ones).

You must define `unbroadcast` AND `sum_back` inline — that's the composition.

The `peel_grad` key is what proves the `unbroadcast-pattern` atom is actually wired up. Without it, an `unbroadcast` that just returns `grad` unchanged would pass all the easy cases.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx27_round_trip(x, dim):
    raise NotImplementedError

def _test_cx27():
    # Case A: 2-D sum over dim=1, keepdim=False.
    x = t.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
    out = cx27_round_trip(x, dim=1)
    assert set(out.keys()) >= {'forward_sum', 'sum_back_grad', 'unbroadcast_grad', 'peel_grad'}, out.keys()
    assert out['forward_sum'].shape == (2,), out['forward_sum'].shape
    assert t.allclose(out['forward_sum'], t.tensor([6.0, 15.0]))
    assert out['sum_back_grad'].shape == x.shape, out['sum_back_grad'].shape
    assert t.allclose(out['sum_back_grad'], t.ones_like(x)), out['sum_back_grad']
    assert out['unbroadcast_grad'].shape == x.shape, out['unbroadcast_grad'].shape
    assert t.allclose(out['unbroadcast_grad'], t.ones_like(x)), out['unbroadcast_grad']
    # Duality: sum_back of ones-seed equals unbroadcast of broadcast-back of ones-seed.
    assert t.allclose(out['sum_back_grad'], out['unbroadcast_grad'])

    # Case B: 3-D sum over dim=0 — leading-axis case.
    x2 = t.randn(4, 3, 5)
    out2 = cx27_round_trip(x2, dim=0)
    assert out2['forward_sum'].shape == (3, 5)
    assert out2['sum_back_grad'].shape == x2.shape
    assert t.allclose(out2['sum_back_grad'], t.ones_like(x2))
    assert t.allclose(out2['sum_back_grad'], out2['unbroadcast_grad'])

    # Case C: atom-coverage — unbroadcast-pattern MUST peel axes.
    # The original test fed unbroadcast a tensor that already matched x.shape,
    # so an unbroadcast that just returned its input would pass. This case
    # constructs a (5,3,4) upstream gradient that was forward-broadcast from
    # (5,1,4) and verifies unbroadcast collapses axis 1 (3 -> 1, summing to 3.0).
    peel = out.get('peel_grad', None)
    assert peel is not None, "cx27_round_trip must include 'peel_grad' in its return dict"
    assert tuple(peel.shape) == (5, 1, 4), (
        f"unbroadcast must peel size-3 -> size-1 on axis 1; got shape {tuple(peel.shape)}. "
        "A pass-through unbroadcast would return shape (5, 3, 4) — this case catches that bug."
    )
    assert t.allclose(peel, t.full((5, 1, 4), 3.0)), (
        f"unbroadcast must SUM the broadcast axis (three ones -> 3.0 per entry); got {peel}"
    )
    # Verify peel_grad result is stable across calls (deterministic, no randomness).
    out_again = cx27_round_trip(x, dim=1)
    assert t.equal(out_again['peel_grad'], peel), 'peel_grad must be deterministic'
    _dd_passed.add('cx27')

_test_cx27()

<details><summary>Show solution — cx27</summary>

```python
def cx27_round_trip(x, dim):
    # Inline unbroadcast: peel leading + collapse size-1 axes.
    def unbroadcast(grad, original):
        while grad.ndim > original.ndim:
            grad = grad.sum(dim=0)
        for i, size in enumerate(original.shape):
            if size == 1 and grad.shape[i] != 1:
                grad = grad.sum(dim=i, keepdim=True)
        return grad

    # Inline sum_back: re-insert the dropped axis, then expand back to x.shape.
    def sum_back(grad_out, out, x_ref, dim, keepdim=False):
        if not keepdim:
            grad_out = grad_out.unsqueeze(dim)
        return grad_out.expand_as(x_ref).clone()

    forward_sum = x.sum(dim=dim, keepdim=False)
    seed = t.ones_like(forward_sum)
    sum_back_grad = sum_back(seed, forward_sum, x, dim, keepdim=False)
    # Broadcast-back path: pretend forward was broadcast_to(x.shape) — unbroadcast collapses it.
    broadcast_back = sum_back_grad.clone()  # already at x.shape via sum_back
    unbroadcast_grad = unbroadcast(broadcast_back, x)

    # Load-bearing case: unbroadcast must actually peel size-3 axis back to size-1.
    # peel_ref has shape (5,1,4); peel_grad_out has shape (5,3,4) (the broadcast-back).
    # unbroadcast must sum axis 1 to collapse 3 -> 1, yielding each entry = 3.0.
    peel_ref = t.zeros(5, 1, 4)
    peel_grad_out = t.ones(5, 3, 4)
    peel_grad = unbroadcast(peel_grad_out, peel_ref)

    return {
        'forward_sum': forward_sum,
        'sum_back_grad': sum_back_grad,
        'unbroadcast_grad': unbroadcast_grad,
        'peel_grad': peel_grad,
    }
```

The two atoms are duals: `sum_back` inserts a size-1 axis and expands; `unbroadcast` collapses leading + size-1 axes by summing. On a uniform-ones seed they agree exactly. The deeper invariant: `sum_back ∘ broadcast == id` on the size-1-axis Jacobian — they are inverse adjoints.

The `peel_grad` fixture forces `unbroadcast` to do real work — a pass-through implementation that returned its input unchanged would yield shape `(5, 3, 4)` and fail. By computing `unbroadcast(t.ones(5,3,4), t.zeros(5,1,4))` and asserting the result is shape `(5, 1, 4)` with each entry summing to 3.0, we lock in the `unbroadcast-pattern` atom semantics.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx27'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx27',
        'subtopics': ["Backprop: sum/broadcast duality", "Backprop: Unbroadcast pattern"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()